In [1]:
# Membuat log terpisah dengan hasil pengujian test tidak tercampur dengan log validasi sebelumnya
from pathlib import Path
import subprocess

PROJECT = Path("/home/rzangef/SSD-PPE")
PYTHON = PROJECT / ".venv/bin/python"
SCRIPT = PROJECT / "object-detection-api/research/object_detection/model_main_tf2.py"
CONFIG = PROJECT / "models/pipeline_ppe_yolo_split_test.config"

CHECKPOINT_DIR = PROJECT / "training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split"
EVAL_DIR = PROJECT / "evaluation/ssd_stage5_test_ckpt7"
LOG = PROJECT / "logs/ssd_stage5_test_ckpt7.log"

EVAL_DIR.mkdir(parents=True, exist_ok=True)
LOG.parent.mkdir(parents=True, exist_ok=True)

# Pastikan file dan checkpoint tersedia
required = [
    PYTHON,
    SCRIPT,
    CONFIG,
    CHECKPOINT_DIR / "ckpt-7.index",
]

missing = [str(p) for p in required if not p.exists()]

if missing:
    print("File berikut belum ditemukan:")
    for p in missing:
        print("-", p)
else:
    cmd = [
        str(PYTHON),
        str(SCRIPT),
        f"--model_dir={EVAL_DIR}",
        f"--pipeline_config_path={CONFIG}",
        f"--checkpoint_dir={CHECKPOINT_DIR}",
        "--eval_timeout=1",
        "--alsologtostderr",
    ]

    log_file = open(LOG, "w")
    process = subprocess.Popen(
        cmd,
        cwd=str(PROJECT),
        stdout=log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    log_file.close()

    print("Evaluasi test SSD diluncurkan.")
    print("PID             :", process.pid)
    print("Checkpoint      : ckpt-7")
    print("Jumlah data test: 340 gambar")
    print("Log             :", LOG)
    print("Direktori hasil :", EVAL_DIR)

Evaluasi test SSD diluncurkan.
PID             : 3126
Checkpoint      : ckpt-7
Jumlah data test: 340 gambar
Log             : /home/rzangef/SSD-PPE/logs/ssd_stage5_test_ckpt7.log
Direktori hasil : /home/rzangef/SSD-PPE/evaluation/ssd_stage5_test_ckpt7


In [2]:
# memeriksa log untuk mengetahui apakah SSD sudah selesai mengevaluasi checkpoint ckpt-7 pada data test
from pathlib import Path
import subprocess

log = Path("/home/rzangef/SSD-PPE/logs/ssd_stage5_test_ckpt7.log")

# Periksa apakah proses evaluasi masih berjalan
result = subprocess.run(
    ["bash", "-lc", "ps -p 3126 -o pid=,stat=,etime=,cmd="],
    capture_output=True,
    text=True
)

print("=== STATUS PROSES ===")
print(result.stdout.strip() or "Proses PID 3126 sudah tidak berjalan.")

print("\n=== 50 BARIS TERAKHIR LOG ===")
if log.exists():
    output = subprocess.run(
        ["tail", "-n", "50", str(log)],
        capture_output=True,
        text=True
    )
    print(output.stdout or "(Log masih kosong)")
else:
    print("File log belum ditemukan.")

=== STATUS PROSES ===
3126 Zs         01:50 [python] <defunct>

=== 50 BARIS TERAKHIR LOG ===
2026-10-23 21:34:39.047280: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-23 21:34:39.752339: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-23 21:34:39.752707: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-23 21:34:39.905249: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-10-23 21:34:40.204193: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU w

In [3]:
# Tidak ditemukan modul object_detection
# Perbaiki Pythonpath
# menguji apakah modul bisa ditemukan terlebih dahulu

from pathlib import Path
import os
import subprocess

PROJECT = Path("/home/rzangef/SSD-PPE")
PYTHON = PROJECT / ".venv/bin/python"
RESEARCH = PROJECT / "object-detection-api/research"
SCRIPT = RESEARCH / "object_detection/model_main_tf2.py"
CONFIG = PROJECT / "models/pipeline_ppe_yolo_split_test.config"
CHECKPOINT_DIR = PROJECT / "training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split"
EVAL_DIR = PROJECT / "evaluation/ssd_stage5_test_ckpt7"
LOG = PROJECT / "logs/ssd_stage5_test_ckpt7.log"

# Tambahkan lokasi API Object Detection ke jalur pencarian modul Python
env = os.environ.copy()
paths = [
    str(RESEARCH),
    str(RESEARCH / "slim"),
    env.get("PYTHONPATH", ""),
]
env["PYTHONPATH"] = os.pathsep.join(p for p in paths if p)

# 1. Uji import terlebih dahulu
check = subprocess.run(
    [
        str(PYTHON),
        "-c",
        "from object_detection import model_lib_v2; "
        "print('Import object_detection BERHASIL')"
    ],
    cwd=str(RESEARCH),
    env=env,
    capture_output=True,
    text=True,
)

print("=== PEMERIKSAAN MODUL ===")
print(check.stdout)
if check.returncode != 0:
    print(check.stderr[-4000:])
else:
    # 2. Jalankan evaluasi hanya jika import berhasil
    EVAL_DIR.mkdir(parents=True, exist_ok=True)
    LOG.parent.mkdir(parents=True, exist_ok=True)

    cmd = [
        str(PYTHON),
        str(SCRIPT),
        f"--model_dir={EVAL_DIR}",
        f"--pipeline_config_path={CONFIG}",
        f"--checkpoint_dir={CHECKPOINT_DIR}",
        "--eval_timeout=1",
        "--alsologtostderr",
    ]

    with open(LOG, "w") as f:
        process = subprocess.Popen(
            cmd,
            cwd=str(RESEARCH),
            env=env,
            stdout=f,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )

    print("Evaluasi diluncurkan kembali.")
    print("PID:", process.pid)
    print("Log:", LOG)

=== PEMERIKSAAN MODUL ===
Import object_detection BERHASIL

Evaluasi diluncurkan kembali.
PID: 3221
Log: /home/rzangef/SSD-PPE/logs/ssd_stage5_test_ckpt7.log


In [4]:
# Memastikan evaluasinya benar-benar berjalan sampai selesai dan menghasilkan metrik
from pathlib import Path
import subprocess

log = Path("/home/rzangef/SSD-PPE/logs/ssd_stage5_test_ckpt7.log")

# Periksa proses evaluasi
status = subprocess.run(
    ["bash", "-lc", "ps -p 3221 -o pid=,stat=,etime=,cmd="],
    capture_output=True,
    text=True
)

print("=== STATUS PROSES EVALUASI ===")
print(status.stdout.strip() or "Proses sudah tidak berjalan.")

# Periksa perkembangan log
print("\n=== 60 BARIS TERAKHIR LOG ===")
if log.exists():
    hasil = subprocess.run(
        ["tail", "-n", "60", str(log)],
        capture_output=True,
        text=True
    )
    print(hasil.stdout or "Log belum berisi output.")
else:
    print("File log tidak ditemukan.")

=== STATUS PROSES EVALUASI ===
3221 Ssl        02:03 /home/rzangef/SSD-PPE/.venv/bin/python /home/rzangef/SSD-PPE/object-detection-api/research/object_detection/model_main_tf2.py --model_dir=/home/rzangef/SSD-PPE/evaluation/ssd_stage5_test_ckpt7 --pipeline_config_path=/home/rzangef/SSD-PPE/models/pipeline_ppe_yolo_split_test.config --checkpoint_dir=/home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split --eval_timeout=1 --alsologtostderr

=== 60 BARIS TERAKHIR LOG ===
    
W1023 21:40:30.475463 136871390224384 deprecation.py:50] From /home/rzangef/SSD-PPE/.venv/lib/python3.10/site-packages/tensorflow/python/autograph/impl/api.py:460: py_func (from tensorflow.python.ops.script_ops) is deprecated and will be removed in a future version.
Instructions for updating:
tf.py_func is deprecated in TF V2. Instead, there are two
    options available in V2.
    - tf.py_function takes a python function which manipulates tf eager
    tensors instead of numpy arrays. It's easy to 

In [5]:
# Meringkas output cell diatas
from pathlib import Path
import subprocess

log = Path("/home/rzangef/SSD-PPE/logs/ssd_stage5_test_ckpt7.log")

print("=== 1. STATUS PROSES ===")
status = subprocess.run(
    ["bash", "-lc", "ps -p 3221 -o pid=,stat=,etime=,cmd="],
    capture_output=True,
    text=True
)
print(status.stdout.strip() or "Proses PID 3221 sudah tidak berjalan.")

print("\n=== 2. BARIS LOG PENTING ===")
if log.exists():
    result = subprocess.run(
        ["grep", "-Ein",
         "error|traceback|exception|evaluation|checkpoint|Average Precision|Average Recall|DetectionBoxes|DONE|Finished|Timed-out|Evaluating",
         str(log)],
        capture_output=True,
        text=True
    )
    lines = result.stdout.strip().splitlines()
    print("\n".join(lines[-50:]) if lines and lines != [""] else "Belum ditemukan baris yang cocok.")

    print("\n=== 3. 15 BARIS TERAKHIR LOG ===")
    tail = subprocess.run(
        ["tail", "-n", "15", str(log)],
        capture_output=True,
        text=True
    )
    print(tail.stdout or "Log masih kosong.")
else:
    print("File log tidak ditemukan.")

=== 1. STATUS PROSES ===
3221 Ssl        03:22 /home/rzangef/SSD-PPE/.venv/bin/python /home/rzangef/SSD-PPE/object-detection-api/research/object_detection/model_main_tf2.py --model_dir=/home/rzangef/SSD-PPE/evaluation/ssd_stage5_test_ckpt7 --pipeline_config_path=/home/rzangef/SSD-PPE/models/pipeline_ppe_yolo_split_test.config --checkpoint_dir=/home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split --eval_timeout=1 --alsologtostderr

=== 2. BARIS LOG PENTING ===
19:WARNING:tensorflow:Expected number of evaluation epochs is 1, but instead encountered `eval_on_train_input_config.num_epochs` = 0. Overwriting `num_epochs` to 1.
20:W1023 21:39:37.980754 136871390224384 model_lib_v2.py:1106] Expected number of evaluation epochs is 1, but instead encountered `eval_on_train_input_config.num_epochs` = 0. Overwriting `num_epochs` to 1.
53:INFO:tensorflow:Waiting for new checkpoint at /home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split
54:I1023 21:39:50.48

In [1]:
# Periksa lokasi data test dan label YOLO untuk mengambil anotasi ground truth dan prediksi SSD
from pathlib import Path

paths = [
    Path("/home/rzangef/spark-data/CHVG-YOLO/images/test"),
    Path("/home/rzangef/spark-data/CHVG-YOLO/labels/test"),
    Path("/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/test.record"),
    Path("/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/label_map.pbtxt"),
    Path("/home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split/ckpt-7.index"),
]

print("=== PEMERIKSAAN FILE DATA TEST ===")

for path in paths:
    print(f"\nLokasi : {path}")
    print(f"Tersedia: {path.exists()}")

    if path.is_dir():
        files = list(path.glob("*"))
        print(f"Jumlah file langsung di folder: {len(files)}")
        print("Contoh:", [p.name for p in files[:3]])

=== PEMERIKSAAN FILE DATA TEST ===

Lokasi : /home/rzangef/spark-data/CHVG-YOLO/images/test
Tersedia: True
Jumlah file langsung di folder: 340
Contoh: ['ppe_1265_jpg.rf.ef571c482b5e10840e33f02f272ee811.jpg', 'ppe_0279_jpg.rf.7af5ff4c53121f3c7aeb993e0dc06a79.jpg', 'ppe_1229_jpg.rf.343275ca6b9ec191b097d353c57597dd.jpg']

Lokasi : /home/rzangef/spark-data/CHVG-YOLO/labels/test
Tersedia: True
Jumlah file langsung di folder: 340
Contoh: ['images_jpg.rf.45bbca3b47c629db30b55fd10469d6f5.txt', '000221_jpg.rf.92483e3414819358a5aa335d941e80a3.txt', 'ppe_0777_jpg.rf.f5379558309b3a486a8ad6c9a99e0561.txt']

Lokasi : /home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/test.record
Tersedia: True

Lokasi : /home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/label_map.pbtxt
Tersedia: True

Lokasi : /home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split/ckpt-7.index
Tersedia: True


In [2]:
# Aturan pengujian : kelas harus sama, IoU ≥ 0,50, dan setiap objek hanya boleh dipasangkan satu kali. Ambang confidence akan ditentukan secara konsisten menggunakan data validasi, bukan dipilih berdasarkan hasil test
# Memastikan label test cocok dengan gambar
# Guna perhitungan TP< FP, FN tidak salah

from pathlib import Path
from collections import Counter

IMAGE_DIR = Path("/home/rzangef/spark-data/CHVG-YOLO/images/test")
LABEL_DIR = Path("/home/rzangef/spark-data/CHVG-YOLO/labels/test")

# Urutan kelas YOLO dari data.yaml
CLASS_NAMES = [
    "person", "vest", "yellow", "white",
    "glass", "blue", "red", "head"
]

images = {
    p.stem: p for p in IMAGE_DIR.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
}
labels = {p.stem: p for p in LABEL_DIR.glob("*.txt")}

missing_labels = sorted(set(images) - set(labels))
orphan_labels = sorted(set(labels) - set(images))
invalid_rows = []
class_counts = Counter()
total_objects = 0
empty_labels = 0

for stem in sorted(set(images) & set(labels)):
    label_file = labels[stem]
    rows = [
        line.strip()
        for line in label_file.read_text().splitlines()
        if line.strip()
    ]

    if not rows:
        empty_labels += 1

    for line_no, row in enumerate(rows, start=1):
        parts = row.split()

        try:
            if len(parts) != 5:
                raise ValueError("Jumlah kolom bukan 5")

            cls = int(parts[0])
            coords = [float(x) for x in parts[1:]]

            if not 0 <= cls < len(CLASS_NAMES):
                raise ValueError(f"ID kelas {cls} di luar rentang")

            if not all(0 <= x <= 1 for x in coords):
                raise ValueError("Koordinat di luar rentang 0–1")

            _, _, w, h = coords
            if w <= 0 or h <= 0:
                raise ValueError("Lebar/tinggi bbox tidak valid")

            class_counts[CLASS_NAMES[cls]] += 1
            total_objects += 1

        except Exception as exc:
            invalid_rows.append((label_file.name, line_no, str(exc)))

print("=== HASIL PEMERIKSAAN LABEL TEST ===")
print("Jumlah gambar              :", len(images))
print("Jumlah file label          :", len(labels))
print("Gambar tanpa label pasangan:", len(missing_labels))
print("Label tanpa gambar pasangan:", len(orphan_labels))
print("File label kosong          :", empty_labels)
print("Baris anotasi tidak valid  :", len(invalid_rows))
print("Total objek berlabel       :", total_objects)

print("\n=== JUMLAH OBJEK PER KELAS ===")
for name in CLASS_NAMES:
    print(f"{name:10s}: {class_counts[name]}")

if missing_labels:
    print("\nContoh gambar tanpa label:", missing_labels[:5])
if orphan_labels:
    print("Contoh label tanpa gambar:", orphan_labels[:5])
if invalid_rows:
    print("Contoh anotasi tidak valid:", invalid_rows[:5])

=== HASIL PEMERIKSAAN LABEL TEST ===
Jumlah gambar              : 340
Jumlah file label          : 340
Gambar tanpa label pasangan: 0
Label tanpa gambar pasangan: 0
File label kosong          : 0
Baris anotasi tidak valid  : 0
Total objek berlabel       : 2385

=== JUMLAH OBJEK PER KELAS ===
person    : 964
vest      : 430
yellow    : 247
white     : 263
glass     : 103
blue      : 145
red       : 82
head      : 151


In [4]:
# Uji Model : mengambil checkpoint ckpt-7
from pathlib import Path
import os
import subprocess

PROJECT = Path("/home/rzangef/SSD-PPE")
RESEARCH = PROJECT / "object-detection-api/research"
PYTHON = PROJECT / ".venv/bin/python"

env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join([
    str(RESEARCH),
    str(RESEARCH / "slim"),
    env.get("PYTHONPATH", "")
])

script = r'''
import tensorflow as tf
from object_detection.utils import config_util
from object_detection.builders import model_builder
import inspect

config_path = "/home/rzangef/SSD-PPE/models/pipeline_ppe_yolo_split_test.config"
checkpoint_dir = "/home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split"
checkpoint = checkpoint_dir + "/ckpt-7"

configs = config_util.get_configs_from_pipeline_file(config_path)

print("=== KONFIGURASI MODEL ===")
print("Jenis model:", configs["model"].WhichOneof("model"))
print("Jumlah kelas:",
      configs["model"].ssd.num_classes)

print("\n=== CHECKPOINT ===")
print("Checkpoint terbaru:", tf.train.latest_checkpoint(checkpoint_dir))
print("ckpt-7 tersedia:", tf.io.gfile.exists(checkpoint + ".index"))

variables = tf.train.list_variables(checkpoint)
print("Jumlah tensor checkpoint:", len(variables))
print("Contoh nama tensor:")
for name, shape in variables[:8]:
    print(name, shape)

print("\n=== FUNGSI PEMBANGUN MODEL ===")
print("Signature:", inspect.signature(model_builder.build))

print("\n=== DATA EVALUASI ===")
reader = configs["eval_input_config"]
print("Label map:", reader.label_map_path)
print("TFRecord:", list(reader.tf_record_input_reader.input_path))
'''

result = subprocess.run(
    [str(PYTHON), "-c", script],
    cwd=str(RESEARCH),
    env=env,
    capture_output=True,
    text=True
)

print(result.stdout)
if result.returncode != 0:
    print("=== ERROR ===")
    print(result.stderr[-3000:])

=== KONFIGURASI MODEL ===
Jenis model: ssd
Jumlah kelas: 8

=== CHECKPOINT ===
Checkpoint terbaru: /home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split/ckpt-7
ckpt-7 tersedia: True
Jumlah tensor checkpoint: 622
Contoh nama tensor:
_CHECKPOINTABLE_OBJECT_GRAPH []
model/_box_predictor/_base_tower_layers_for_heads/box_encodings/0/1/beta/.ATTRIBUTES/VARIABLE_VALUE [128]
model/_box_predictor/_base_tower_layers_for_heads/box_encodings/0/1/beta/.OPTIMIZER_SLOT/optimizer/momentum/.ATTRIBUTES/VARIABLE_VALUE [128]
model/_box_predictor/_base_tower_layers_for_heads/box_encodings/0/1/gamma/.ATTRIBUTES/VARIABLE_VALUE [128]
model/_box_predictor/_base_tower_layers_for_heads/box_encodings/0/1/gamma/.OPTIMIZER_SLOT/optimizer/momentum/.ATTRIBUTES/VARIABLE_VALUE [128]
model/_box_predictor/_base_tower_layers_for_heads/box_encodings/0/1/moving_mean/.ATTRIBUTES/VARIABLE_VALUE [128]
model/_box_predictor/_base_tower_layers_for_heads/box_encodings/0/1/moving_variance/.ATTRIBUTES/VARIABLE_V

In [5]:
# Uji prediksi pada 1 gambar

from pathlib import Path
import os
import subprocess

PROJECT = Path("/home/rzangef/SSD-PPE")
RESEARCH = PROJECT / "object-detection-api/research"
PYTHON = PROJECT / ".venv/bin/python"

script = r'''
import glob
import numpy as np
import tensorflow as tf
from object_detection.utils import config_util
from object_detection.builders import model_builder

config_path = "/home/rzangef/SSD-PPE/models/pipeline_ppe_yolo_split_test.config"
checkpoint = "/home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split/ckpt-7"
image_dir = "/home/rzangef/spark-data/CHVG-YOLO/images/test"

images = sorted(glob.glob(image_dir + "/*.jpg"))
if not images:
    images = sorted(glob.glob(image_dir + "/*.JPG"))
if not images:
    raise FileNotFoundError("Gambar JPG tidak ditemukan di folder test.")

print("Gambar uji:", images[0])

configs = config_util.get_configs_from_pipeline_file(config_path)
model = model_builder.build(
    model_config=configs["model"],
    is_training=False
)

# Muat bobot dari checkpoint yang sudah dilatih
status = tf.train.Checkpoint(model=model).restore(checkpoint)
status.expect_partial()

# Jalankan inferensi pada satu gambar
image = tf.io.decode_image(
    tf.io.read_file(images[0]),
    channels=3,
    expand_animations=False
)
input_tensor = tf.cast(image, tf.float32)[tf.newaxis, ...]

preprocessed, shapes = model.preprocess(input_tensor)
prediction_dict = model.predict(preprocessed, shapes)
detections = model.postprocess(prediction_dict, shapes)

boxes = detections["detection_boxes"][0].numpy()
scores = detections["detection_scores"][0].numpy()
classes = detections["detection_classes"][0].numpy().astype(int)

print("\n=== HASIL INFERENSI ===")
print("Ukuran gambar:", image.shape)
print("Jumlah prediksi yang dihasilkan:", len(scores))
print("Jumlah prediksi dengan confidence >= 0.30:",
      int(np.sum(scores >= 0.30)))

print("\nLima prediksi teratas:")
for i in np.argsort(scores)[::-1][:5]:
    print(
        f"Kelas ID: {classes[i]}, "
        f"confidence: {scores[i]:.4f}, "
        f"box [ymin, xmin, ymax, xmax]: "
        f"{np.round(boxes[i], 4).tolist()}"
    )

print("\nUJI INFERENSI SELESAI")
'''

env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join([
    str(RESEARCH),
    str(RESEARCH / "slim"),
    env.get("PYTHONPATH", "")
])

result = subprocess.run(
    [str(PYTHON), "-c", script],
    cwd=str(RESEARCH),
    env=env,
    capture_output=True,
    text=True
)

print(result.stdout)
if result.returncode != 0:
    print("=== ERROR ===")
    print(result.stderr[-4000:])

Gambar uji: /home/rzangef/spark-data/CHVG-YOLO/images/test/000019_jpg.rf.549923f46def56cf8c294def971cc7d9.jpg

=== HASIL INFERENSI ===
Ukuran gambar: (640, 640, 3)
Jumlah prediksi yang dihasilkan: 100
Jumlah prediksi dengan confidence >= 0.30: 1

Lima prediksi teratas:
Kelas ID: 0, confidence: 0.3639, box [ymin, xmin, ymax, xmax]: [0.11209999769926071, 0.29010000824928284, 1.0, 0.5993000268936157]
Kelas ID: 0, confidence: 0.2636, box [ymin, xmin, ymax, xmax]: [0.16040000319480896, 0.39089998602867126, 1.0, 0.7131999731063843]
Kelas ID: 1, confidence: 0.2621, box [ymin, xmin, ymax, xmax]: [0.40790000557899475, 0.23800000548362732, 0.6208999752998352, 0.33980000019073486]
Kelas ID: 0, confidence: 0.2470, box [ymin, xmin, ymax, xmax]: [0.3109999895095825, 0.33889999985694885, 0.8831999897956848, 0.5648999810218811]
Kelas ID: 0, confidence: 0.2311, box [ymin, xmin, ymax, xmax]: [0.18479999899864197, 0.5856000185012817, 1.0, 0.9233999848365784]

UJI INFERENSI SELESAI



In [6]:
# Cek kesesuaiaan dataset SSD dan YOLO

from pathlib import Path

PROJECT = Path("/home/rzangef/SSD-PPE")
YOLO_ROOT = Path("/home/rzangef/spark-data/CHVG-YOLO")
TFRECORD_ROOT = PROJECT / "dataset/tfrecord_yolo_split"

print("=== DATASET YOLO ===")
for split in ["train", "val", "test"]:
    images = list((YOLO_ROOT / "images" / split).glob("*.jpg"))
    images += list((YOLO_ROOT / "images" / split).glob("*.JPG"))
    labels = list((YOLO_ROOT / "labels" / split).glob("*.txt"))
    print(f"{split}: {len(images)} gambar, {len(labels)} label")

print("\n=== TFRECORD SSD ===")
for split in ["train", "val", "test"]:
    path = TFRECORD_ROOT / f"{split}.record"
    print(f"{split}: {path}")
    print("  File tersedia:", path.exists())
    if path.exists():
        print("  Ukuran:", round(path.stat().st_size / (1024 * 1024), 2), "MB")

=== DATASET YOLO ===
train: 1189 gambar, 1189 label
val: 170 gambar, 170 label
test: 340 gambar, 340 label

=== TFRECORD SSD ===
train: /home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/train.record
  File tersedia: True
  Ukuran: 69.75 MB
val: /home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/val.record
  File tersedia: True
  Ukuran: 9.78 MB
test: /home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/test.record
  File tersedia: True
  Ukuran: 19.99 MB


In [7]:
# Baca metadata dari TFRecord

from pathlib import Path
import os
import subprocess

PROJECT = Path("/home/rzangef/SSD-PPE")
PYTHON = PROJECT / ".venv/bin/python"
TFRECORD_ROOT = PROJECT / "dataset/tfrecord_yolo_split"

script = r'''
import tensorflow as tf
from pathlib import Path

root = Path("/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split")

for split in ["train", "val", "test"]:
    path = root / f"{split}.record"
    print(f"\n{'='*15} {split.upper()} {'='*15}")

    if not path.exists():
        print("ERROR: File tidak ditemukan:", path)
        continue

    count = 0
    total_objects = 0
    class_counts = {}
    filenames = []
    errors = []

    for raw in tf.data.TFRecordDataset(str(path)):
        example = tf.train.Example()
        example.ParseFromString(raw.numpy())
        features = example.features.feature

        # Tampilkan struktur fitur dari record pertama
        if count == 0:
            print("Nama fitur:", sorted(features.keys()))

        filename = ""
        for key in ["image/filename", "image/source_id"]:
            if key in features and features[key].bytes_list.value:
                filename = features[key].bytes_list.value[0].decode(
                    "utf-8", errors="replace"
                )
                break

        if filename:
            filenames.append(Path(filename).name)

        # Baca kelas objek
        class_key = None
        for key in [
            "image/object/class/label",
            "image/object/class/text"
        ]:
            if key in features:
                class_key = key
                break

        labels = []
        if class_key:
            feat = features[class_key]
            if class_key.endswith("/label"):
                labels = list(feat.int64_list.value)
            else:
                labels = [
                    x.decode("utf-8", errors="replace")
                    for x in feat.bytes_list.value
                ]

        total_objects += len(labels)
        for label in labels:
            class_counts[label] = class_counts.get(label, 0) + 1

        # Periksa jumlah koordinat bounding box
        for key in [
            "image/object/bbox/xmin",
            "image/object/bbox/xmax",
            "image/object/bbox/ymin",
            "image/object/bbox/ymax"
        ]:
            if key not in features:
                errors.append(f"Record {count}: fitur {key} tidak ada")
                break

        count += 1

    print("Jumlah record/gambar:", count)
    print("Jumlah objek:", total_objects)
    print("Jumlah nama file tercatat:", len(filenames))
    print("Jumlah nama file unik:", len(set(filenames)))
    print("Contoh nama file:", filenames[:5])
    print("Jumlah objek per kelas:", class_counts)
    print("Jumlah masalah struktur bbox:", len(errors))

    if errors:
        print("Contoh masalah:", errors[:3])
'''

env = os.environ.copy()

result = subprocess.run(
    [str(PYTHON), "-c", script],
    capture_output=True,
    text=True
)

print(result.stdout)
if result.returncode != 0:
    print("=== ERROR ===")
    print(result.stderr[-3000:])


=============== TRAIN ===============
Nama fitur: ['image/encoded', 'image/filename', 'image/format', 'image/height', 'image/object/bbox/xmax', 'image/object/bbox/xmin', 'image/object/bbox/ymax', 'image/object/bbox/ymin', 'image/object/class/label', 'image/object/class/text', 'image/object/difficult', 'image/object/truncated', 'image/source_id', 'image/width']
Jumlah record/gambar: 1189
Jumlah objek: 8008
Jumlah nama file tercatat: 1189
Jumlah nama file unik: 1189
Contoh nama file: ['000004_jpg.rf.a64a312cbd6f217ceb7680f99531fcd6.jpg', '000091_jpg.rf.75339227b3c4f06e011096c7fa4e1ff5.jpg', '000132_jpg.rf.39acd7bca3d210ca59deafbe0204b1d8.jpg', '000196_jpg.rf.a4b74f2e3d652164086fb2c1cfbe741f.jpg', '000207_jpg.rf.38f3a56ce761fc3529d467134a323b81.jpg']
Jumlah objek per kelas: {8: 487, 1: 3220, 5: 380, 4: 899, 3: 841, 6: 343, 2: 1474, 7: 364}
Jumlah masalah struktur bbox: 0

=============== VAL ===============
Nama fitur: ['image/encoded', 'image/filename', 'image/format', 'image/height', '

In [8]:
# Untuk menguji SSD menghasilkan Precision, Recall, F1-Score 
# 1. Periksa pemetaan kelas SSD
# 2. Gunakan data validasi (170 gambar) untuk menentukan ambang confidence (confidence threshold) yang sesuai
# 3. Setelah ambang ditentukan, evaluasi data test (340 gambar) satu kali untuk menghitung Precision, Recall, dan F1-score
# 4. Bandingkan hasil SSD dan YOLO menggunakan data test yang sama serta aturan penghitungan yang konsisten

In [9]:
from pathlib import Path
import subprocess
import os

PROJECT = Path("/home/rzangef/SSD-PPE")
PYTHON = PROJECT / ".venv/bin/python"
RESEARCH = PROJECT / "object-detection-api/research"

CONFIG = PROJECT / "models/pipeline_ppe_yolo_split_val.config"
CHECKPOINT_DIR = (
    PROJECT / "training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split"
)
LABEL_MAP = PROJECT / "dataset/tfrecord_yolo_split/label_map.pbtxt"

print("Python tersedia :", PYTHON.exists())
print("Config tersedia :", CONFIG.exists())
print("Checkpoint dir  :", CHECKPOINT_DIR.exists())
print("Label map       :", LABEL_MAP.exists())

if LABEL_MAP.exists():
    print("\n=== LABEL MAP ===")
    print(LABEL_MAP.read_text())

env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join([
    str(RESEARCH),
    str(RESEARCH / "slim"),
    env.get("PYTHONPATH", "")
])

script = r"""
from object_detection.utils import config_util
import tensorflow as tf
import sys

config_path = sys.argv[1]
checkpoint_dir = sys.argv[2]

configs = config_util.get_configs_from_pipeline_file(config_path)
print("\n=== KONFIGURASI MODEL ===")
print("Jenis model:", configs["model"].WhichOneof("model"))
print("Jumlah kelas:", configs["model"].ssd.num_classes)

print("\n=== CHECKPOINT ===")
print("Checkpoint terbaru:", tf.train.latest_checkpoint(checkpoint_dir))
"""

result = subprocess.run(
    [
        str(PYTHON), "-c", script,
        str(CONFIG), str(CHECKPOINT_DIR)
    ],
    cwd=str(RESEARCH),
    env=env,
    capture_output=True,
    text=True
)

print("\n=== OUTPUT PEMERIKSAAN ===")
print(result.stdout)
if result.stderr:
    print("\n=== PESAN ERROR (jika ada) ===")
    print(result.stderr[-3000:])

Python tersedia : True
Config tersedia : True
Checkpoint dir  : True
Label map       : True

=== LABEL MAP ===
item {
    id: 1
    name: 'person'
}

item {
    id: 2
    name: 'vest'
}

item {
    id: 3
    name: 'yellow'
}

item {
    id: 4
    name: 'white'
}

item {
    id: 5
    name: 'glass'
}

item {
    id: 6
    name: 'blue'
}

item {
    id: 7
    name: 'red'
}

item {
    id: 8
    name: 'head'
}



=== OUTPUT PEMERIKSAAN ===

=== KONFIGURASI MODEL ===
Jenis model: ssd
Jumlah kelas: 8

=== CHECKPOINT ===
Checkpoint terbaru: /home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split/ckpt-7


=== PESAN ERROR (jika ada) ===
2026-10-24 07:22:12.336987: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-24 07:22:13.071927: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been 

In [10]:
# Periksa ID kelas hasil prediksi
from pathlib import Path
import os
import subprocess

PROJECT = Path("/home/rzangef/SSD-PPE")
PYTHON = PROJECT / ".venv/bin/python"
RESEARCH = PROJECT / "object-detection-api/research"
CONFIG = PROJECT / "models/pipeline_ppe_yolo_split_val.config"
CHECKPOINT_DIR = PROJECT / "training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split"

IMAGE = Path(
    "/home/rzangef/spark-data/CHVG-YOLO/images/test/"
    "000019_jpg.rf.549923f46def56cf8c294def971cc7d9.jpg"
)

script = r'''
import tensorflow as tf
import numpy as np
from object_detection.utils import config_util
from object_detection.builders import model_builder
import sys

config_path, checkpoint_dir, image_path = sys.argv[1:4]

configs = config_util.get_configs_from_pipeline_file(config_path)
model = model_builder.build(
    model_config=configs["model"],
    is_training=False
)

ckpt = tf.train.latest_checkpoint(checkpoint_dir)
tf.train.Checkpoint(model=model).restore(ckpt).expect_partial()

image = tf.io.decode_image(
    tf.io.read_file(image_path),
    channels=3,
    expand_animations=False
)
tensor = tf.cast(image, tf.float32)[tf.newaxis, ...]
preprocessed, shapes = model.preprocess(tensor)
predictions = model.predict(preprocessed, shapes)
detections = model.postprocess(predictions, shapes)

scores = detections["detection_scores"][0].numpy()
classes = detections["detection_classes"][0].numpy().astype(int)

print("Checkpoint:", ckpt)
print("ID kelas dengan confidence >= 0.10:",
      sorted(set(classes[scores >= 0.10].tolist())))
print("ID kelas dengan confidence >= 0.30:",
      sorted(set(classes[scores >= 0.30].tolist())))
print("\nLima prediksi teratas:")
for i in np.argsort(scores)[::-1][:5]:
    print("Kelas:", int(classes[i]),
          "| Confidence:", round(float(scores[i]), 4))
'''

env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join([
    str(RESEARCH),
    str(RESEARCH / "slim"),
    env.get("PYTHONPATH", "")
])

result = subprocess.run(
    [
        str(PYTHON), "-c", script,
        str(CONFIG), str(CHECKPOINT_DIR), str(IMAGE)
    ],
    cwd=str(RESEARCH),
    env=env,
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print("Pesan error:\n", result.stderr[-3000:])

Checkpoint: /home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split/ckpt-7
ID kelas dengan confidence >= 0.10: [0, 1, 2, 3, 7]
ID kelas dengan confidence >= 0.30: [0]

Lima prediksi teratas:
Kelas: 0 | Confidence: 0.3639
Kelas: 0 | Confidence: 0.2636
Kelas: 1 | Confidence: 0.2621
Kelas: 0 | Confidence: 0.247
Kelas: 0 | Confidence: 0.2311

Pesan error:
 2026-10-24 07:24:45.793014: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-24 07:24:46.498954: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-24 07:24:46.499379: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-24 07:24:46.638263: E external/local_xla/xla/stream_e

In [11]:
# memeriksa bagaimana model SSD mengeluarkan kelas dan bagaimana konfigurasi labelnya dibuat
import os
import subprocess
from pathlib import Path

PROJECT = Path("/home/rzangef/SSD-PPE")
PYTHON = PROJECT / ".venv/bin/python"
RESEARCH = PROJECT / "object-detection-api/research"
CONFIG = PROJECT / "models/pipeline_ppe_yolo_split_val.config"

script = r'''
from object_detection.utils import config_util
import sys

configs = config_util.get_configs_from_pipeline_file(sys.argv[1])
ssd = configs["model"].ssd

print("Jumlah kelas:", ssd.num_classes)
print("add_background_class:", ssd.add_background_class)
print("\nKonfigurasi label dan preprocessing:")
print("label_map_path:", configs["eval_input_config"].label_map_path)
print("input_path:", list(configs["eval_input_config"].input_path))
'''

env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join([
    str(RESEARCH), str(RESEARCH / "slim"),
    env.get("PYTHONPATH", "")
])

result = subprocess.run(
    [str(PYTHON), "-c", script, str(CONFIG)],
    cwd=str(RESEARCH), env=env,
    capture_output=True, text=True
)

print(result.stdout)
if result.stderr:
    print(result.stderr[-2000:])

Jumlah kelas: 8
add_background_class: True

Konfigurasi label dan preprocessing:
label_map_path: /home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/label_map.pbtxt

2026-10-24 07:26:25.001731: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-24 07:26:25.092042: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-24 07:26:25.092163: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-24 07:26:25.095952: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-10-24 07:26:25.112546: I external/local_tsl/tsl/cuda

In [12]:
# Memeriksa pemetaan kelas pada konfigurasi model dan kode SSD yang terpasang
import inspect
from pathlib import Path
import subprocess
import os

PROJECT = Path("/home/rzangef/SSD-PPE")
PYTHON = PROJECT / ".venv/bin/python"
RESEARCH = PROJECT / "object-detection-api/research"

script = r'''
import inspect
from object_detection.models import ssd_meta_arch

source = inspect.getsource(ssd_meta_arch.SSDMetaArch.postprocess)

print("=== BAGIAN POSTPROCESS SSD ===")
lines = source.splitlines()

matches = [
    i for i, line in enumerate(lines)
    if "detection_classes" in line or "classes" in line and (
        "cast" in line or "+ 1" in line or "add" in line
    )
]

shown = set()
for i in matches:
    for j in range(max(0, i-3), min(len(lines), i+5)):
        if j not in shown:
            print(f"{j+1}: {lines[j]}")
            shown.add(j)
    print("---")
'''

env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join([
    str(RESEARCH),
    str(RESEARCH / "slim"),
    env.get("PYTHONPATH", "")
])

result = subprocess.run(
    [str(PYTHON), "-c", script],
    cwd=str(RESEARCH),
    env=env,
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print("Pesan tambahan:\n", result.stderr[-1500:])


Pesan tambahan:
 Traceback (most recent call last):
  File "<string>", line 3, in <module>
ImportError: cannot import name 'ssd_meta_arch' from 'object_detection.models' (/home/rzangef/SSD-PPE/object-detection-api/research/object_detection/models/__init__.py)



In [13]:
# Cari kode pemetaan kelas
from pathlib import Path

RESEARCH = Path(
    "/home/rzangef/SSD-PPE/object-detection-api/research"
)

# Cari file yang mendefinisikan SSDMetaArch
matches = []

for file in RESEARCH.rglob("*.py"):
    try:
        content = file.read_text(errors="ignore")
        if "class SSDMetaArch" in content:
            matches.append(file)
    except OSError:
        pass

print("File yang mendefinisikan SSDMetaArch:")
for file in matches:
    print(file)

# Cari bagian kode yang menghasilkan detection_classes
for file in matches:
    lines = file.read_text(errors="ignore").splitlines()
    print(f"\n=== {file} ===")

    found = False
    for i, line in enumerate(lines):
        if "detection_classes" in line:
            found = True
            start = max(0, i - 4)
            end = min(len(lines), i + 5)

            for j in range(start, end):
                print(f"{j+1}: {lines[j]}")
            print("---")

    if not found:
        print("Belum ditemukan teks detection_classes pada file ini.")

File yang mendefinisikan SSDMetaArch:
/home/rzangef/SSD-PPE/object-detection-api/research/object_detection/meta_architectures/ssd_meta_arch_test_lib.py
/home/rzangef/SSD-PPE/object-detection-api/research/object_detection/meta_architectures/ssd_meta_arch.py

=== /home/rzangef/SSD-PPE/object-detection-api/research/object_detection/meta_architectures/ssd_meta_arch_test_lib.py ===
Belum ditemukan teks detection_classes pada file ini.

=== /home/rzangef/SSD-PPE/object-detection-api/research/object_detection/meta_architectures/ssd_meta_arch.py ===
313:       non_max_suppression_fn: batch_multiclass_non_max_suppression
314:         callable that takes `boxes`, `scores` and optional `clip_window`
315:         inputs (with all other inputs already set) and returns a dictionary
316:         hold tensors with keys: `detection_boxes`, `detection_scores`,
317:         `detection_classes` and `num_detections`. See `post_processing.
318:         batch_multiclass_non_max_suppression` for the type and 

In [14]:
# Cek apakah kelas 0 berasal dari pemrosesan prediksi SD atau konfigurasi model
from pathlib import Path

FILE = Path(
    "/home/rzangef/SSD-PPE/object-detection-api/research/"
    "object_detection/meta_architectures/ssd_meta_arch.py"
)

lines = FILE.read_text().splitlines()

print("=== SSD POST-PROCESSING ===")
for i in range(735, 790):
    print(f"{i+1}: {lines[i]}")

=== SSD POST-PROCESSING ===
736: 
737:       batch_size = (
738:           shape_utils.combined_static_and_dynamic_shape(preprocessed_images)[0])
739: 
740:       if 'feature_maps' in prediction_dict:
741:         feature_map_list = []
742:         for feature_map in prediction_dict['feature_maps']:
743:           feature_map_list.append(tf.reshape(feature_map, [batch_size, -1]))
744:         box_features = tf.concat(feature_map_list, 1)
745:         box_features = tf.identity(box_features, 'raw_box_features')
746:       additional_fields = {
747:           'multiclass_scores': detection_scores_with_background
748:       }
749:       if self._anchors is not None:
750:         num_boxes = (self._anchors.num_boxes_static() or
751:                      self._anchors.num_boxes())
752:         anchor_indices = tf.range(num_boxes)
753:         batch_anchor_indices = tf.tile(
754:             tf.expand_dims(anchor_indices, 0), [batch_size, 1])
755:         # All additional fields need to be f

In [15]:
# Cek fungsi NMS yang menghasilkan ID Class dimulai dari 0
# Seharusnya dimulai dari 1
from pathlib import Path

RESEARCH = Path(
    "/home/rzangef/SSD-PPE/object-detection-api/research"
)

# Cari implementasi fungsi NMS yang digunakan SSD
for file in RESEARCH.rglob("*.py"):
    try:
        lines = file.read_text(errors="ignore").splitlines()
    except OSError:
        continue

    for i, line in enumerate(lines):
        if "def batch_multiclass_non_max_suppression" in line:
            print(f"\n=== FILE: {file} ===")
            start = max(0, i - 5)
            end = min(len(lines), i + 100)

            for j in range(start, end):
                print(f"{j+1}: {lines[j]}")


=== FILE: /home/rzangef/SSD-PPE/object-detection-api/research/object_detection/core/post_processing.py ===
873:                                          tf.range(num_valid_nms_boxes))
874: 
875:     return sorted_boxes, num_valid_nms_boxes
876: 
877: 
878: def batch_multiclass_non_max_suppression(boxes,
879:                                          scores,
880:                                          score_thresh,
881:                                          iou_thresh,
882:                                          max_size_per_class,
883:                                          max_total_size=0,
884:                                          clip_window=None,
885:                                          change_coordinate_frame=False,
886:                                          num_valid_boxes=None,
887:                                          masks=None,
888:                                          additional_fields=None,
889:                                          soft_nms_

In [16]:
# Pemeriksaan kecil untuk melihat bagaimana konfigurasi SSD mengatur kelas latar belakang dan skor kelas saat inferensi
import os
import subprocess
from pathlib import Path

PROJECT = Path("/home/rzangef/SSD-PPE")
PYTHON = PROJECT / ".venv/bin/python"
RESEARCH = PROJECT / "object-detection-api/research"
CONFIG = PROJECT / "models/pipeline_ppe_yolo_split_val.config"

script = r'''
from object_detection.utils import config_util
import sys

cfg = config_util.get_configs_from_pipeline_file(sys.argv[1])
ssd = cfg["model"].ssd

print("num_classes:", ssd.num_classes)
print("add_background_class:", ssd.add_background_class)
print("score_converter:", ssd.post_processing.score_converter)
print("use_bfloat16:", ssd.use_bfloat16)
'''

env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join([
    str(RESEARCH), str(RESEARCH / "slim"),
    env.get("PYTHONPATH", "")
])

result = subprocess.run(
    [str(PYTHON), "-c", script, str(CONFIG)],
    cwd=str(RESEARCH), env=env,
    capture_output=True, text=True
)

print(result.stdout)
if result.stderr:
    print(result.stderr[-1500:])

num_classes: 8
add_background_class: True
score_converter: 1

.479060: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-24 07:42:46.626149: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-10-24 07:42:46.921136: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-24 07:42:46.924188: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/rzangef/SSD-PPE/.venv/lib/python3.10/site-packages/google/api_core/_pyt

In [17]:
# Melihat konfigurasi pemetaan kelas pada proses NMS
from pathlib import Path

FILE = Path(
    "/home/rzangef/SSD-PPE/object-detection-api/research/"
    "object_detection/core/post_processing.py"
)

lines = FILE.read_text().splitlines()

# Tampilkan implementasi NMS, bukan dokumentasinya
start = next(
    i for i, line in enumerate(lines)
    if line.startswith("def batch_multiclass_non_max_suppression")
)

for i in range(start + 100, min(start + 230, len(lines))):
    print(f"{i+1}: {lines[i]}")

978:     'nmsed_classes': A [batch_size, max_detections] float32 tensor
979:       containing the class for boxes.
980:     'nmsed_masks': (optional) a
981:       [batch_size, max_detections, mask_height, mask_width] float32 tensor
982:       containing masks for each selected box. This is set to None if input
983:       `masks` is None.
984:     'nmsed_additional_fields': (optional) a dictionary of
985:       [batch_size, max_detections, ...] float32 tensors corresponding to the
986:       tensors specified in the input `additional_fields`. This is not returned
987:       if input `additional_fields` is None.
988:     'num_detections': A [batch_size] int32 tensor indicating the number of
989:       valid detections per batch item. Only the top num_detections[i] entries in
990:       nms_boxes[i], nms_scores[i] and nms_class[i] are valid. The rest of the
991:       entries are zero paddings.
992: 
993:   Raises:
994:     ValueError: if `q` in boxes.shape is not 1 or not equal to number

In [18]:
from pathlib import Path

FILE = Path(
    "/home/rzangef/SSD-PPE/object-detection-api/research/"
    "object_detection/core/post_processing.py"
)

lines = FILE.read_text().splitlines()

for i in range(1098, min(1240, len(lines))):
    print(f"{i+1}: {lines[i]}")

1099:     def _single_image_nms_fn(args):
1100:       """Runs NMS on a single image and returns padded output.
1101: 
1102:       Args:
1103:         args: A list of tensors consisting of the following:
1104:           per_image_boxes - A [num_anchors, q, 4] float32 tensor containing
1105:             detections. If `q` is 1 then same boxes are used for all classes
1106:             otherwise, if `q` is equal to number of classes, class-specific
1107:             boxes are used.
1108:           per_image_scores - A [num_anchors, num_classes] float32 tensor
1109:             containing the scores for each of the `num_anchors` detections.
1110:           per_image_masks - A [num_anchors, q, mask_height, mask_width] float32
1111:             tensor containing box masks. `q` can be either number of classes
1112:             or 1 depending on whether a separate mask is predicted per class.
1113:           per_image_clip_window - A 1D float32 tensor of the form
1114:             [ymin, xmin,

In [19]:
# Periksa fungsi sebelum memastikan nmsed_classes mengambil ID kelas dari hasil multiclass_non_max_suppression
from pathlib import Path

FILE = Path(
    "/home/rzangef/SSD-PPE/object-detection-api/research/"
    "object_detection/core/post_processing.py"
)

lines = FILE.read_text().splitlines()

start = next(
    i for i, line in enumerate(lines)
    if line.startswith("def multiclass_non_max_suppression")
)

for i in range(start, min(start + 180, len(lines))):
    print(f"{i+1}: {lines[i]}")

422: def multiclass_non_max_suppression(boxes,
423:                                    scores,
424:                                    score_thresh,
425:                                    iou_thresh,
426:                                    max_size_per_class,
427:                                    max_total_size=0,
428:                                    clip_window=None,
429:                                    change_coordinate_frame=False,
430:                                    masks=None,
431:                                    boundaries=None,
432:                                    pad_to_max_output_size=False,
433:                                    use_partitioned_nms=False,
434:                                    additional_fields=None,
435:                                    soft_nms_sigma=0.0,
436:                                    use_hard_nms=False,
437:                                    use_cpu_nms=False,
438:                                    scope=None):
439:   """

In [21]:
# Melihat bagian yang menetapkan ID kelas pada setiap prediksi
for i in range(595, min(700, len(lines))):
    print(f"{i+1}: {lines[i]}")

596:                tf.zeros(max_selection_size-num_valid_nms_boxes,
597:                         tf.float32)], -1)
598:           nms_result = box_list_ops.gather(boxlist_and_class_scores,
599:                                            selected_indices)
600:         else:
601:           if soft_nms_sigma != 0:
602:             raise ValueError('Soft NMS not supported in current TF version!')
603:           selected_indices = tf.image.non_max_suppression(
604:               boxlist_and_class_scores.get(),
605:               boxlist_and_class_scores.get_field(fields.BoxListFields.scores),
606:               max_selection_size,
607:               iou_threshold=iou_thresh,
608:               score_threshold=score_thresh)
609:           num_valid_nms_boxes = tf.shape(selected_indices)[0]
610:           selected_indices = tf.concat(
611:               [selected_indices,
612:                tf.zeros(max_selection_size-num_valid_nms_boxes, tf.int32)], 0)
613:           nms_result = box_list_

In [22]:
# Dari output diatas cari add_field(fields.BoxListFields.classes), class_idx, classes, + 1 atau class_idx + 1
keywords = [
    "BoxListFields.classes",
    "class_idx",
    "add_field",
    "classes",
]

for i in range(594, min(750, len(lines))):
    if any(k in lines[i] for k in keywords):
        print(f"{i+1}: {lines[i]}")

620:       nms_result.add_field(
626:       nms_result.add_field(
627:           fields.BoxListFields.classes, (tf.zeros_like(
628:               nms_result.get_field(fields.BoxListFields.scores)) + class_idx))
658:                                        max_classes_per_detection=1,
674:   max scores across all classes for which scores are provided (via the scores
679:   therefore any background classes should be removed prior to calling this
687:       number of classes or 1 depending on whether a separate box is predicted
689:     scores: A [k, num_classes] float32 tensor containing the scores for each of
695:     max_classes_per_detection: maximum number of retained classes per detection
697:     max_total_size: maximum number of boxes retained over all classes. By
706:       containing box masks. `q` can be either number of classes or 1 depending
709:       tensor containing box boundaries. `q` can be either number of classes or 1
730:       and a rank-1 classes field representing 

In [23]:
# Memeriksa konfigurasi kelas SSD
print("num_classes:",
      configs["model"].ssd.num_classes)

print("add_background_class:",
      configs["model"].ssd.encode_background_as_zeros)

print("score_threshold:",
      configs["model"].ssd.post_processing
      .batch_non_max_suppression.score_threshold)

NameError: name 'configs' is not defined

In [24]:
from pathlib import Path
from google.protobuf import text_format
from object_detection.protos import pipeline_pb2

CONFIG_PATH = Path(
    "/home/rzangef/SSD-PPE/models/"
    "pipeline_ppe_yolo_split_val.config"
)

pipeline_config = pipeline_pb2.TrainEvalPipelineConfig()

with CONFIG_PATH.open("r") as f:
    text_format.Merge(f.read(), pipeline_config)

ssd_config = pipeline_config.model.ssd

print("=== KONFIGURASI KELAS SSD ===")
print("num_classes:", ssd_config.num_classes)
print("encode_background_as_zeros:",
      ssd_config.encode_background_as_zeros)
print("score_threshold:",
      ssd_config.post_processing
      .batch_non_max_suppression.score_threshold)

ModuleNotFoundError: No module named 'object_detection'

In [25]:
# Menambahkan lokasi modul object_detection ke python
import sys
from pathlib import Path

RESEARCH_DIR = Path(
    "/home/rzangef/SSD-PPE/object-detection-api/research"
)

print("Folder research tersedia:", RESEARCH_DIR.exists())

if str(RESEARCH_DIR) not in sys.path:
    sys.path.insert(0, str(RESEARCH_DIR))

import object_detection

print("Import object_detection berhasil")
print("Lokasi modul:", object_detection.__file__)

Folder research tersedia: True
Import object_detection berhasil
Lokasi modul: /home/rzangef/SSD-PPE/object-detection-api/research/object_detection/__init__.py


In [26]:
# Berhasil, lanjut memeriksa konfigurasi SSD
from pathlib import Path
from google.protobuf import text_format
from object_detection.protos import pipeline_pb2

CONFIG_PATH = Path(
    "/home/rzangef/SSD-PPE/models/"
    "pipeline_ppe_yolo_split_val.config"
)

pipeline_config = pipeline_pb2.TrainEvalPipelineConfig()

with CONFIG_PATH.open("r") as f:
    text_format.Merge(f.read(), pipeline_config)

ssd_config = pipeline_config.model.ssd

print("num_classes:", ssd_config.num_classes)
print("encode_background_as_zeros:",
      ssd_config.encode_background_as_zeros)
print("score_threshold:",
      ssd_config.post_processing
      .batch_non_max_suppression.score_threshold)

num_classes: 8
encode_background_as_zeros: True
score_threshold: 9.99999993922529e-09


In [27]:
# Melakukan inferensi 1 gambar
# Untuk periksa ID kelas hasil deteksi
import tensorflow as tf
import numpy as np

IMAGE_PATH = (
    "/home/rzangef/spark-data/CHVG-YOLO/images/test/"
    "000019_jpg.rf.549923f46def56cf8c294def971cc7d9.jpg"
)

CHECKPOINT = (
    "/home/rzangef/SSD-PPE/training/"
    "ssd_mobilenet_v2_fpnlite_320x320_yolo_split/ckpt-7"
)

print("TensorFlow:", tf.__version__)
print("Gambar tersedia:", tf.io.gfile.exists(IMAGE_PATH))
print("Checkpoint tersedia:",
      tf.io.gfile.exists(CHECKPOINT + ".index"))

2026-10-24 08:05:51.144640: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-24 08:05:51.871544: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-24 08:05:51.871919: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-24 08:05:52.015691: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-10-24 08:05:52.319545: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-24 08:05:52.327155: I tensorflow/core/platform/cpu_feature_guard.cc:1

TensorFlow: 2.15.1
Gambar tersedia: True
Checkpoint tersedia: True


In [28]:
# membangun model SSD dan mengambil hasil deteksinya menggunakan konfigurasi validasi
import sys
from pathlib import Path
import tensorflow as tf
import numpy as np
from google.protobuf import text_format

RESEARCH_DIR = "/home/rzangef/SSD-PPE/object-detection-api/research"
if RESEARCH_DIR not in sys.path:
    sys.path.insert(0, RESEARCH_DIR)

from object_detection.protos import pipeline_pb2
from object_detection.builders import model_builder

CONFIG_PATH = (
    "/home/rzangef/SSD-PPE/models/"
    "pipeline_ppe_yolo_split_val.config"
)
CHECKPOINT = (
    "/home/rzangef/SSD-PPE/training/"
    "ssd_mobilenet_v2_fpnlite_320x320_yolo_split/ckpt-7"
)
IMAGE_PATH = (
    "/home/rzangef/spark-data/CHVG-YOLO/images/test/"
    "000019_jpg.rf.549923f46def56cf8c294def971cc7d9.jpg"
)

pipeline_config = pipeline_pb2.TrainEvalPipelineConfig()
with open(CONFIG_PATH, "r") as f:
    text_format.Merge(f.read(), pipeline_config)

detection_model = model_builder.build(
    model_config=pipeline_config.model,
    is_training=False
)

checkpoint = tf.train.Checkpoint(model=detection_model)
checkpoint.restore(CHECKPOINT).expect_partial()

image = tf.io.decode_image(
    tf.io.read_file(IMAGE_PATH),
    channels=3,
    expand_animations=False
)
image = tf.cast(image, tf.float32)[tf.newaxis, ...]

preprocessed_image, shapes = detection_model.preprocess(image)
prediction_dict = detection_model.predict(preprocessed_image, shapes)
detections = detection_model.postprocess(prediction_dict, shapes)

boxes = detections["detection_boxes"][0].numpy()
scores = detections["detection_scores"][0].numpy()
classes = detections["detection_classes"][0].numpy().astype(int)

valid = scores >= 0.10

print("Jumlah prediksi confidence >= 0.10:", int(valid.sum()))
print("ID kelas unik:", np.unique(classes[valid]).tolist())
print("\n10 prediksi teratas:")
for i in np.argsort(scores)[::-1][:10]:
    if scores[i] <= 0:
        continue
    print(f"ID kelas={classes[i]}, confidence={scores[i]:.4f}")

Jumlah prediksi confidence >= 0.10: 93
ID kelas unik: [0, 1, 2, 3, 7]

10 prediksi teratas:
ID kelas=0, confidence=0.3639
ID kelas=0, confidence=0.2636
ID kelas=1, confidence=0.2621
ID kelas=0, confidence=0.2470
ID kelas=0, confidence=0.2311
ID kelas=0, confidence=0.2141
ID kelas=0, confidence=0.2113
ID kelas=1, confidence=0.2051
ID kelas=0, confidence=0.1951
ID kelas=0, confidence=0.1941


In [29]:
# Apakah prediksi SSD dan label ground truth menggunakan sistem ID kelas yang sama
# Memeriksa label kelas asli pada test.record dan mencocokannya dengan label_map.pbtxt
from pathlib import Path
import tensorflow as tf
import re

BASE = Path("/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split")
LABEL_MAP = BASE / "label_map.pbtxt"
TEST_RECORD = BASE / "test.record"

# Baca nama kelas dari label map
label_text = LABEL_MAP.read_text()
label_map = {
    int(class_id): name
    for class_id, name in re.findall(
        r"item\s*\{\s*id:\s*(\d+)\s*name:\s*['\"]([^'\"]+)['\"]",
        label_text,
        re.DOTALL
    )
}

print("=== LABEL MAP ===")
for class_id, name in sorted(label_map.items()):
    print(f"ID {class_id}: {name}")

# Cari gambar yang sama di test.record
target = "000019_jpg.rf.549923f46def56cf8c294def971cc7d9.jpg"

feature_spec = {
    "image/filename": tf.io.FixedLenFeature([], tf.string, default_value=""),
    "image/object/class/label": tf.io.VarLenFeature(tf.int64),
}

found = False

for raw_record in tf.data.TFRecordDataset([str(TEST_RECORD)]):
    example = tf.io.parse_single_example(raw_record, feature_spec)
    filename = example["image/filename"].numpy().decode(
        "utf-8", errors="replace"
    )

    if target in filename:
        gt_ids = tf.sparse.to_dense(
            example["image/object/class/label"]
        ).numpy().tolist()

        print("\n=== GROUND TRUTH GAMBAR ===")
        print("Nama file:", filename)
        print("ID kelas ground truth:", sorted(set(gt_ids)))
        print("Nama kelas ground truth:", sorted({
            label_map.get(i, f"ID tidak dikenal: {i}") for i in gt_ids
        }))
        found = True
        break

if not found:
    print("\nGambar tidak ditemukan di test.record.")
    print("Kita akan periksa format nama file pada record berikutnya.")

=== LABEL MAP ===
ID 1: person
ID 2: vest
ID 3: yellow
ID 4: white
ID 5: glass
ID 6: blue
ID 7: red
ID 8: head

=== GROUND TRUTH GAMBAR ===
Nama file: 000019_jpg.rf.549923f46def56cf8c294def971cc7d9.jpg
ID kelas ground truth: [1, 8]
Nama kelas ground truth: ['head', 'person']


In [30]:
# Uji pemetaan kelas pada validasi dengan ID prediksi SSD + 1 = ID kelas pada label map
from pathlib import Path
import tensorflow as tf

VAL_RECORD = Path(
    "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/val.record"
)

feature_spec = {
    "image/filename": tf.io.FixedLenFeature([], tf.string, default_value=""),
    "image/object/class/label": tf.io.VarLenFeature(tf.int64),
}

for raw_record in tf.data.TFRecordDataset([str(VAL_RECORD)]).take(1):
    example = tf.io.parse_single_example(raw_record, feature_spec)

    filename = example["image/filename"].numpy().decode(
        "utf-8", errors="replace"
    )
    gt_ids = tf.sparse.to_dense(
        example["image/object/class/label"]
    ).numpy().tolist()

    print("Nama gambar:", filename)
    print("ID ground truth:", sorted(set(gt_ids)))
    print("ID prediksi yang perlu dipetakan: +1")
    print("Contoh: prediksi 0 -> label 1 (person)")

Nama gambar: 000006_jpg.rf.ad2eab92c4615adc64a5c4358f816dfa.jpg
ID ground truth: [1, 8]
ID prediksi yang perlu dipetakan: +1
Contoh: prediksi 0 -> label 1 (person)


In [31]:
# Mencari bagian postprocess yang menangani skor kelas untuk kita tambahkan prediksi ssd + 1
from pathlib import Path

FILE = Path(
    "/home/rzangef/SSD-PPE/object-detection-api/research/"
    "object_detection/meta_architectures/ssd_meta_arch.py"
)

lines = FILE.read_text().splitlines()

keywords = [
    "class_predictions_with_background",
    "class_predictions =",
    "add_background_class",
    "background",
]

for i, line in enumerate(lines):
    if any(k in line for k in keywords):
        print(f"\n--- Sekitar baris {i+1} ---")
        for j in range(max(0, i-3), min(len(lines), i+5)):
            print(f"{j+1}: {lines[j]}")


--- Sekitar baris 267 ---
264:                box_predictor,
265:                box_coder,
266:                feature_extractor,
267:                encode_background_as_zeros,
268:                image_resizer_fn,
269:                non_max_suppression_fn,
270:                score_conversion_fn,
271:                classification_loss,

--- Sekitar baris 282 ---
279:                normalize_loc_loss_by_codesize=False,
280:                freeze_batchnorm=False,
281:                inplace_batchnorm_update=False,
282:                add_background_class=True,
283:                explicit_background_class=False,
284:                random_example_sampler=None,
285:                expected_loss_weights_fn=None,
286:                use_confidences_as_targets=False,

--- Sekitar baris 283 ---
280:                freeze_batchnorm=False,
281:                inplace_batchnorm_update=False,
282:                add_background_class=True,
283:                explicit_background_class=False

In [32]:
# Tampilkan out diatas hanya bagian 

from pathlib import Path

FILE = Path(
    "/home/rzangef/SSD-PPE/object-detection-api/research/"
    "object_detection/meta_architectures/ssd_meta_arch.py"
)

lines = FILE.read_text().splitlines()

keywords = [
    "class_predictions_with_background",
    "class_predictions_without_background",
    "_batch_non_max_suppression",
    "class_predictions =",
]

for i, line in enumerate(lines):
    if any(k in line for k in keywords):
        print(f"\n--- Baris {i+1} ---")
        start = max(0, i - 8)
        end = min(len(lines), i + 12)
        for j in range(start, end):
            print(f"{j+1}: {lines[j]}")


--- Baris 549 ---
541:         with zeros.
542: 
543:     Returns:
544:       prediction_dict: a dictionary holding "raw" prediction tensors:
545:         1) preprocessed_inputs: the [batch, height, width, channels] image
546:           tensor.
547:         2) box_encodings: 4-D float tensor of shape [batch_size, num_anchors,
548:           box_code_dimension] containing predicted boxes.
549:         3) class_predictions_with_background: 3-D float tensor of shape
550:           [batch_size, num_anchors, num_classes+1] containing class predictions
551:           (logits) for each of the anchors.  Note that this tensor *includes*
552:           background class predictions (at class index 0).
553:         4) feature_maps: a list of tensors where the ith tensor has shape
554:           [batch, height_i, width_i, depth_i].
555:         5) anchors: 2-D float tensor of shape [num_anchors, 4] containing
556:           the generated anchors in normalized coordinates.
557:         6) final_anc

In [33]:
# Memeriksa pengaturan background dan pemanggilan NMS pada postprocess
from pathlib import Path

FILE = Path(
    "/home/rzangef/SSD-PPE/object-detection-api/research/"
    "object_detection/meta_architectures/ssd_meta_arch.py"
)

lines = FILE.read_text().splitlines()

# Tampilkan bagian postprocess, dari baris 729 sampai sekitar 790
for i in range(728, min(790, len(lines))):
    print(f"{i+1}: {lines[i]}")

print("\n=== PENGATURAN BACKGROUND ===")
for i, line in enumerate(lines):
    if "self._add_background_class =" in line or \
       "self._explicit_background_class =" in line:
        print(f"\nBaris {i+1}:")
        for j in range(max(0, i-2), min(len(lines), i+5)):
            print(f"{j+1}: {lines[j]}")

729:       detection_scores_with_background = self._score_conversion_fn(
730:           class_predictions_with_background)
731:       detection_scores = tf.identity(detection_scores_with_background,
732:                                      'raw_box_scores')
733:       if self._add_background_class or self._explicit_background_class:
734:         detection_scores = tf.slice(detection_scores, [0, 0, 1], [-1, -1, -1])
735:       additional_fields = None
736: 
737:       batch_size = (
738:           shape_utils.combined_static_and_dynamic_shape(preprocessed_images)[0])
739: 
740:       if 'feature_maps' in prediction_dict:
741:         feature_map_list = []
742:         for feature_map in prediction_dict['feature_maps']:
743:           feature_map_list.append(tf.reshape(feature_map, [batch_size, -1]))
744:         box_features = tf.concat(feature_map_list, 1)
745:         box_features = tf.identity(box_features, 'raw_box_features')
746:       additional_fields = {
747:           'multicl

In [34]:

print("=== PENGATURAN BACKGROUND MODEL SSD ===")

print(
    "add_background_class:",
    pipeline_config.model.ssd.add_background_class
)

print(
    "explicit_background_class:",
    pipeline_config.model.ssd.explicit_background_class
)

=== PENGATURAN BACKGROUND MODEL SSD ===
add_background_class: True
explicit_background_class: False


In [35]:
# Periksa pemetaan ID kelas di NMS

from pathlib import Path

FILE = Path(
    "/home/rzangef/SSD-PPE/object-detection-api/research/"
    "object_detection/core/post_processing.py"
)

lines = FILE.read_text().splitlines()

# Cari bagian yang memberikan ID kelas pada hasil NMS
for i, line in enumerate(lines):
    if "BoxListFields.classes" in line:
        print(f"\n=== Baris {i+1} ===")
        for j in range(max(0, i-7), min(len(lines), i+8)):
            print(f"{j+1}: {lines[j]}")


=== Baris 627 ===
620:       nms_result.add_field(
621:           fields.BoxListFields.scores,
622:           tf.where(valid_nms_boxes_indices,
623:                    selected_scores, -1*tf.ones(max_selection_size)))
624:       num_valid_nms_boxes_cumulative += num_valid_nms_boxes
625: 
626:       nms_result.add_field(
627:           fields.BoxListFields.classes, (tf.zeros_like(
628:               nms_result.get_field(fields.BoxListFields.scores)) + class_idx))
629:       selected_boxes_list.append(nms_result)
630:     selected_boxes = box_list_ops.concatenate(selected_boxes_list)
631:     sorted_boxes = box_list_ops.sort_by_field(selected_boxes,
632:                                               fields.BoxListFields.scores)
633:     if clip_window is not None:
634:       # When pad_to_max_output_size is False, it prunes the boxes with zero

=== Baris 853 ===
846:     nms_result.add_field(
847:         fields.BoxListFields.scores,
848:         tf.where(valid_nms_boxes_indices,
849:  

In [36]:
# Melihat fungsi NMS yang dipanggil dair postprocess
# Untuk memastikan jalur mana yang dipakai model SSD saya

from pathlib import Path

FILE = Path(
    "/home/rzangef/SSD-PPE/object-detection-api/research/"
    "object_detection/meta_architectures/ssd_meta_arch.py"
)

lines = FILE.read_text().splitlines()

for i in range(730, min(790, len(lines))):
    print(f"{i+1}: {lines[i]}")

731:       detection_scores = tf.identity(detection_scores_with_background,
732:                                      'raw_box_scores')
733:       if self._add_background_class or self._explicit_background_class:
734:         detection_scores = tf.slice(detection_scores, [0, 0, 1], [-1, -1, -1])
735:       additional_fields = None
736: 
737:       batch_size = (
738:           shape_utils.combined_static_and_dynamic_shape(preprocessed_images)[0])
739: 
740:       if 'feature_maps' in prediction_dict:
741:         feature_map_list = []
742:         for feature_map in prediction_dict['feature_maps']:
743:           feature_map_list.append(tf.reshape(feature_map, [batch_size, -1]))
744:         box_features = tf.concat(feature_map_list, 1)
745:         box_features = tf.identity(box_features, 'raw_box_features')
746:       additional_fields = {
747:           'multiclass_scores': detection_scores_with_background
748:       }
749:       if self._anchors is not None:
750:         num_boxes 

In [37]:
print("=== KONFIGURASI NMS MODEL SSD ===")

print(
    "Fungsi NMS:",
    detection_model._non_max_suppression_fn
)

print("\n=== NAMA FUNGSI NMS ===")
print(
    getattr(
        detection_model._non_max_suppression_fn,
        "__name__",
        "Nama fungsi tidak tersedia"
    )
)

=== KONFIGURASI NMS MODEL SSD ===
Fungsi NMS: functools.partial(<function batch_multiclass_non_max_suppression at 0x7a34f2edcdc0>, score_thresh=9.99999993922529e-09, iou_thresh=0.6000000238418579, max_size_per_class=100, max_total_size=100, use_static_shapes=False, use_class_agnostic_nms=False, max_classes_per_detection=1, soft_nms_sigma=0.0, use_partitioned_nms=False, use_combined_nms=False, change_coordinate_frame=True, use_hard_nms=False, use_cpu_nms=False)

=== NAMA FUNGSI NMS ===
Nama fungsi tidak tersedia


In [38]:
import inspect
from object_detection.core import post_processing

print("=== FUNGSI NMS ===")

source = inspect.getsource(
    post_processing.batch_multiclass_non_max_suppression
)
lines = source.splitlines()

keywords = [
    "multiclass_non_max_suppression",
    "class_idx",
    "classes_with_max_scores",
    "scores",
]

shown = set()

for i, line in enumerate(lines):
    if any(k in line for k in keywords):
        start = max(0, i - 3)
        end = min(len(lines), i + 5)

        if any(j in shown for j in range(start, end)):
            continue

        print(f"\n--- Bagian fungsi, baris {i+1} ---")
        for j in range(start, end):
            print(f"{j+1}: {lines[j]}")
            shown.add(j)

=== FUNGSI NMS ===

--- Bagian fungsi, baris 1 ---
1: def batch_multiclass_non_max_suppression(boxes,
2:                                          scores,
3:                                          score_thresh,
4:                                          iou_thresh,
5:                                          max_size_per_class,

--- Bagian fungsi, baris 25 ---
22:                                          use_cpu_nms=False):
23:   """Multi-class version of non maximum suppression that operates on a batch.
24: 
25:   This op is similar to `multiclass_non_max_suppression` but operates on a batch
26:   of boxes and scores. See documentation for `multiclass_non_max_suppression`
27:   for details.
28: 
29:   Args:

--- Bagian fungsi, baris 34 ---
31:       detections. If `q` is 1 then same boxes are used for all classes
32:       otherwise, if `q` is equal to number of classes, class-specific boxes are
33:       used.
34:     scores: A [batch_size, num_anchors, num_classes] float32 tensor 

In [1]:
# Evaluasi validasi SSD
from pathlib import Path

print("=== PEMERIKSAAN SEBELUM EVALUASI ===")

print("Model tersedia:", "detection_model" in globals())
print("Konfigurasi tersedia:", "pipeline_config" in globals())

CHECKPOINT = Path(
    "/home/rzangef/SSD-PPE/training/"
    "ssd_mobilenet_v2_fpnlite_320x320_yolo_split/ckpt-7"
)

VAL_RECORD = Path(
    "/home/rzangef/SSD-PPE/dataset/"
    "tfrecord_yolo_split/val.record"
)

LABEL_MAP = Path(
    "/home/rzangef/SSD-PPE/dataset/"
    "tfrecord_yolo_split/label_map.pbtxt"
)

for name, path in [
    ("Checkpoint", CHECKPOINT),
    ("TFRecord validasi", VAL_RECORD),
    ("Label map", LABEL_MAP),
]:
    print(f"{name} tersedia:", path.exists(), "|", path)

=== PEMERIKSAAN SEBELUM EVALUASI ===
Model tersedia: False
Konfigurasi tersedia: False
Checkpoint tersedia: False | /home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split/ckpt-7
TFRecord validasi tersedia: True | /home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/val.record
Label map tersedia: True | /home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/label_map.pbtxt


In [2]:
# Cari checkpoint SSD
from pathlib import Path

PROJECT = Path("/home/rzangef/SSD-PPE")
TRAINING_DIR = PROJECT / "training"

print("=== FOLDER TRAINING ===")
print("Folder tersedia:", TRAINING_DIR.exists())

if TRAINING_DIR.exists():
    for path in sorted(TRAINING_DIR.rglob("*")):
        if path.is_file() and (
            path.name.startswith("ckpt-")
            or path.name == "checkpoint"
            or path.name.endswith(".index")
            or path.name.endswith(".data-00000-of-00001")
        ):
            print(path)

=== FOLDER TRAINING ===
Folder tersedia: True
/home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320/checkpoint
/home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320/ckpt-1.data-00000-of-00001
/home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320/ckpt-1.index
/home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_20epoch/checkpoint
/home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_20epoch/ckpt-18.data-00000-of-00001
/home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_20epoch/ckpt-18.index
/home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_20epoch/ckpt-19.data-00000-of-00001
/home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_20epoch/ckpt-19.index
/home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_20epoch/ckpt-20.data-00000-of-00001
/home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_20epoch/ckpt-20.index
/home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_20epoc

In [4]:
# Pastikan konfigurasi yang sesuai
from pathlib import Path

PROJECT = Path("/home/rzangef/SSD-PPE")

paths = {
    "Konfigurasi training": PROJECT / "models/pipeline_ppe_yolo_split_stage5.config",
    "Konfigurasi validasi": PROJECT / "models/pipeline_ppe_yolo_split_val.config",
    "Checkpoint SSD": PROJECT / "training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split/ckpt-7.index",
    "Data validasi": PROJECT / "dataset/tfrecord_yolo_split/val.record",
    "Label map": PROJECT / "dataset/tfrecord_yolo_split/label_map.pbtxt",
}

for name, path in paths.items():
    print(f"{name}: {'TERSEDIA' if path.exists() else 'TIDAK DITEMUKAN'}")
    print(f"  {path}")

Konfigurasi training: TERSEDIA
  /home/rzangef/SSD-PPE/models/pipeline_ppe_yolo_split_stage5.config
Konfigurasi validasi: TERSEDIA
  /home/rzangef/SSD-PPE/models/pipeline_ppe_yolo_split_val.config
Checkpoint SSD: TERSEDIA
  /home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split/ckpt-7.index
Data validasi: TERSEDIA
  /home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/val.record
Label map: TERSEDIA
  /home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/label_map.pbtxt


In [5]:
# Muat model SSD
import sys
import tensorflow as tf
from pathlib import Path
from google.protobuf import text_format

RESEARCH_DIR = (
    "/home/rzangef/SSD-PPE/object-detection-api/research"
)
if RESEARCH_DIR not in sys.path:
    sys.path.insert(0, RESEARCH_DIR)

from object_detection.protos import pipeline_pb2
from object_detection.builders import model_builder

PROJECT = Path("/home/rzangef/SSD-PPE")

CONFIG_PATH = PROJECT / "models/pipeline_ppe_yolo_split_val.config"
CHECKPOINT_PATH = (
    PROJECT / "training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split/ckpt-7"
)

# Baca konfigurasi validasi
pipeline_config = pipeline_pb2.TrainEvalPipelineConfig()
with open(CONFIG_PATH, "r") as f:
    text_format.Merge(f.read(), pipeline_config)

# Bangun arsitektur model tanpa training
detection_model = model_builder.build(
    model_config=pipeline_config.model,
    is_training=False
)

# Muat bobot dari checkpoint yang sudah dilatih
checkpoint = tf.train.Checkpoint(model=detection_model)
restore_status = checkpoint.restore(str(CHECKPOINT_PATH))
restore_status.expect_partial()

print("=== MODEL SSD BERHASIL DIMUAT ===")
print("Konfigurasi:", CONFIG_PATH)
print("Checkpoint:", CHECKPOINT_PATH)
print("Jumlah kelas:", pipeline_config.model.ssd.num_classes)
print("Add background class:",
      pipeline_config.model.ssd.add_background_class)
print("TensorFlow:", tf.__version__)

2026-10-24 14:40:38.083661: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-24 14:40:37.753137: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-24 14:40:37.753467: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-24 14:40:37.908857: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-10-24 14:40:38.223361: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-24 14:40:38.226769: I tensorflow/core/platform/cpu_feature_guard.cc:1

=== MODEL SSD BERHASIL DIMUAT ===
Konfigurasi: /home/rzangef/SSD-PPE/models/pipeline_ppe_yolo_split_val.config
Checkpoint: /home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split/ckpt-7
Jumlah kelas: 8
Add background class: True
TensorFlow: 2.15.1


In [6]:
# Uji satu gambar pertama dari val.record menjalankan inferensi SSD lalu membandingkan dengan ID kelas ground truth
import tensorflow as tf
import numpy as np

VAL_RECORD = (
    "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/val.record"
)

features = {
    "image/encoded": tf.io.FixedLenFeature([], tf.string),
    "image/filename": tf.io.FixedLenFeature([], tf.string, default_value=b""),
    "image/object/bbox/xmin": tf.io.VarLenFeature(tf.float32),
    "image/object/bbox/xmax": tf.io.VarLenFeature(tf.float32),
    "image/object/bbox/ymin": tf.io.VarLenFeature(tf.float32),
    "image/object/bbox/ymax": tf.io.VarLenFeature(tf.float32),
    "image/object/class/label": tf.io.VarLenFeature(tf.int64),
}

# Ambil satu contoh dari TFRecord validasi
raw_record = next(iter(tf.data.TFRecordDataset(VAL_RECORD)))
example = tf.io.parse_single_example(raw_record, features)

image_bytes = example["image/encoded"]
filename = example["image/filename"].numpy().decode(
    "utf-8", errors="replace"
)

image = tf.io.decode_image(
    image_bytes, channels=3, expand_animations=False
)
image = tf.cast(image, tf.float32)[tf.newaxis, ...]

# Inferensi menggunakan model SSD yang sudah dimuat
preprocessed_image, true_shapes = detection_model.preprocess(image)
prediction_dict = detection_model.predict(
    preprocessed_image, true_shapes
)
detections = detection_model.postprocess(
    prediction_dict, true_shapes
)

num_det = int(detections["num_detections"][0].numpy())
pred_boxes = detections["detection_boxes"][0].numpy()[:num_det]
pred_scores = detections["detection_scores"][0].numpy()[:num_det]
pred_classes_raw = (
    detections["detection_classes"][0].numpy()[:num_det].astype(int)
)

# Ground truth dari TFRecord
def sparse_values(key, dtype):
    return tf.sparse.to_dense(example[key]).numpy().astype(dtype)

gt_labels = sparse_values("image/object/class/label", np.int32)
gt_xmin = sparse_values("image/object/bbox/xmin", np.float32)
gt_xmax = sparse_values("image/object/bbox/xmax", np.float32)
gt_ymin = sparse_values("image/object/bbox/ymin", np.float32)
gt_ymax = sparse_values("image/object/bbox/ymax", np.float32)

print("=== UJI SATU GAMBAR VALIDASI ===")
print("Filename:", filename)
print("Ukuran gambar (tinggi, lebar):", image.shape[1:3])
print("Jumlah ground truth:", len(gt_labels))
print("ID kelas ground truth:", sorted(set(gt_labels.tolist())))
print("Jumlah deteksi valid dari model:", num_det)

print("\n=== 10 DETEKSI DENGAN SKOR TERTINGGI ===")
order = np.argsort(-pred_scores)
for i in order[:10]:
    print(
        f"score={pred_scores[i]:.4f}, "
        f"class_raw={pred_classes_raw[i]}, "
        f"class_plus_1={pred_classes_raw[i] + 1}, "
        f"box_yxyx={np.round(pred_boxes[i], 3).tolist()}"
    )

print("\nJumlah prediksi dengan confidence >= 0.10:",
      int(np.sum(pred_scores >= 0.10)))

=== UJI SATU GAMBAR VALIDASI ===
Filename: 000006_jpg.rf.ad2eab92c4615adc64a5c4358f816dfa.jpg
Ukuran gambar (tinggi, lebar): (640, 640)
Jumlah ground truth: 9
ID kelas ground truth: [1, 8]
Jumlah deteksi valid dari model: 100

=== 10 DETEKSI DENGAN SKOR TERTINGGI ===
score=0.4982, class_raw=0, class_plus_1=1, box_yxyx=[0.1589999943971634, 0.3400000035762787, 0.949999988079071, 0.7549999952316284]
score=0.4061, class_raw=0, class_plus_1=1, box_yxyx=[0.2160000056028366, 0.2590000033378601, 0.6589999794960022, 0.414000004529953]
score=0.3769, class_raw=0, class_plus_1=1, box_yxyx=[0.24300000071525574, 0.4009999930858612, 0.8650000095367432, 0.7009999752044678]
score=0.2709, class_raw=0, class_plus_1=1, box_yxyx=[0.29499998688697815, 0.28200000524520874, 0.5960000157356262, 0.38999998569488525]
score=0.2669, class_raw=1, class_plus_1=2, box_yxyx=[0.1589999943971634, 0.3400000035762787, 0.949999988079071, 0.7549999952316284]
score=0.2531, class_raw=0, class_plus_1=1, box_yxyx=[0.11699999868

In [7]:
# Evaluasi precision, recall, F1-Score pada data validasi
import tensorflow as tf
import numpy as np
import pandas as pd
import time

VAL_RECORD = (
    "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/val.record"
)

FEATURES = {
    "image/encoded": tf.io.FixedLenFeature([], tf.string),
    "image/filename": tf.io.FixedLenFeature([], tf.string, default_value=b""),
    "image/object/bbox/xmin": tf.io.VarLenFeature(tf.float32),
    "image/object/bbox/xmax": tf.io.VarLenFeature(tf.float32),
    "image/object/bbox/ymin": tf.io.VarLenFeature(tf.float32),
    "image/object/bbox/ymax": tf.io.VarLenFeature(tf.float32),
    "image/object/class/label": tf.io.VarLenFeature(tf.int64),
}

def sparse_array(example, key, dtype):
    return tf.sparse.to_dense(example[key]).numpy().astype(dtype)

def calculate_iou(box, boxes):
    # Format box: [ymin, xmin, ymax, xmax]
    if len(boxes) == 0:
        return np.array([], dtype=np.float32)

    y1 = np.maximum(box[0], boxes[:, 0])
    x1 = np.maximum(box[1], boxes[:, 1])
    y2 = np.minimum(box[2], boxes[:, 2])
    x2 = np.minimum(box[3], boxes[:, 3])

    intersection = np.maximum(0, y2-y1) * np.maximum(0, x2-x1)

    area_box = max(0, box[2]-box[0]) * max(0, box[3]-box[1])
    area_boxes = (
        np.maximum(0, boxes[:, 2]-boxes[:, 0]) *
        np.maximum(0, boxes[:, 3]-boxes[:, 1])
    )

    union = area_box + area_boxes - intersection
    return intersection / np.maximum(union, 1e-9)

# Simpan hasil inferensi agar model hanya dijalankan sekali per gambar.
records = []
total_gt = 0
start_time = time.time()

dataset = tf.data.TFRecordDataset(VAL_RECORD)

for image_index, raw in enumerate(dataset):
    example = tf.io.parse_single_example(raw, FEATURES)

    image = tf.io.decode_image(
        example["image/encoded"],
        channels=3,
        expand_animations=False
    )
    image = tf.cast(image, tf.float32)[tf.newaxis, ...]

    preprocessed, shapes = detection_model.preprocess(image)
    prediction = detection_model.predict(preprocessed, shapes)
    detections = detection_model.postprocess(prediction, shapes)

    n = int(detections["num_detections"][0].numpy())

    pred_boxes = detections["detection_boxes"][0].numpy()[:n]
    pred_scores = detections["detection_scores"][0].numpy()[:n]

    # NMS mengembalikan ID foreground mulai dari 0.
    pred_classes = (
        detections["detection_classes"][0].numpy()[:n].astype(int) + 1
    )

    gt_labels = sparse_array(
        example, "image/object/class/label", np.int32
    )
    xmin = sparse_array(example, "image/object/bbox/xmin", np.float32)
    xmax = sparse_array(example, "image/object/bbox/xmax", np.float32)
    ymin = sparse_array(example, "image/object/bbox/ymin", np.float32)
    ymax = sparse_array(example, "image/object/bbox/ymax", np.float32)

    # Ground truth diubah ke format [ymin, xmin, ymax, xmax].
    gt_boxes = np.column_stack([ymin, xmin, ymax, xmax]).astype(np.float32)

    if not (
        len(gt_labels) == len(gt_boxes)
        == len(xmin) == len(xmax) == len(ymin) == len(ymax)
    ):
        raise ValueError(f"Jumlah anotasi tidak cocok pada gambar {image_index}")

    records.append({
        "pred_boxes": pred_boxes,
        "pred_scores": pred_scores,
        "pred_classes": pred_classes,
        "gt_boxes": gt_boxes,
        "gt_classes": gt_labels,
    })

    total_gt += len(gt_labels)

    if (image_index + 1) % 25 == 0:
        print(f"Sudah diproses: {image_index + 1} gambar")

print("\n=== INFERENSI VALIDASI SELESAI ===")
print("Jumlah gambar:", len(records))
print("Jumlah ground truth:", total_gt)
print(f"Waktu inferensi: {(time.time()-start_time)/60:.1f} menit")

# Evaluasi ulang hasil inferensi pada berbagai confidence threshold.
confidence_thresholds = [
    0.05, 0.10, 0.15, 0.20, 0.25,
    0.30, 0.35, 0.40, 0.50
]
IOU_THRESHOLD = 0.50

results = []

for conf_threshold in confidence_thresholds:
    TP = FP = FN = 0

    for rec in records:
        gt_boxes = rec["gt_boxes"]
        gt_classes = rec["gt_classes"]

        keep = rec["pred_scores"] >= conf_threshold
        boxes = rec["pred_boxes"][keep]
        scores = rec["pred_scores"][keep]
        classes = rec["pred_classes"][keep]

        # Urutkan prediksi dari confidence tertinggi.
        order = np.argsort(-scores)
        boxes = boxes[order]
        classes = classes[order]

        matched_gt = set()
        image_tp = 0

        for box, pred_class in zip(boxes, classes):
            # Hanya GT dengan kelas yang sama yang dapat dicocokkan.
            candidate_indices = [
                j for j, gt_class in enumerate(gt_classes)
                if gt_class == pred_class and j not in matched_gt
            ]

            if not candidate_indices:
                continue

            candidate_boxes = gt_boxes[candidate_indices]
            ious = calculate_iou(box, candidate_boxes)
            best_pos = int(np.argmax(ious))

            if ious[best_pos] >= IOU_THRESHOLD:
                matched_gt.add(candidate_indices[best_pos])
                image_tp += 1

        TP += image_tp
        FP += len(boxes) - image_tp
        FN += len(gt_boxes) - image_tp

    precision = TP / (TP + FP) if TP + FP else 0.0
    recall = TP / (TP + FN) if TP + FN else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    results.append({
        "Confidence": conf_threshold,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
    })

validation_results = pd.DataFrame(results)

print("\n=== HASIL EVALUASI DATA VALIDASI ===")
display(
    validation_results.style.format({
        "Confidence": "{:.2f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1-score": "{:.4f}",
    })
)

best_row = validation_results.loc[
    validation_results["F1-score"].idxmax()
]

print("\n=== AMBANG CONFIDENCE TERBAIK PADA VALIDASI ===")
print("Confidence:", best_row["Confidence"])
print("Precision:", f"{best_row['Precision']:.4f}")
print("Recall:", f"{best_row['Recall']:.4f}")
print("F1-score:", f"{best_row['F1-score']:.4f}")
print("TP:", int(best_row["TP"]))
print("FP:", int(best_row["FP"]))
print("FN:", int(best_row["FN"]))

Sudah diproses: 25 gambar
Sudah diproses: 50 gambar
Sudah diproses: 75 gambar
Sudah diproses: 100 gambar
Sudah diproses: 125 gambar
Sudah diproses: 150 gambar

=== INFERENSI VALIDASI SELESAI ===
Jumlah gambar: 170
Jumlah ground truth: 1211
Waktu inferensi: 1.7 menit

=== HASIL EVALUASI DATA VALIDASI ===


AttributeError: The '.style' accessor requires jinja2

In [8]:
# Atasi eror jinja2
# Tampilkan hasil evaluasi tanpa menggunakan .style
print("=== HASIL EVALUASI DATA VALIDASI ===")
display(validation_results)

# Ambil hasil dengan F1-score tertinggi
best_row = validation_results.loc[
    validation_results["F1-score"].idxmax()
]

print("\n=== AMBANG CONFIDENCE TERBAIK PADA VALIDASI ===")
print("Confidence:", best_row["Confidence"])
print("Precision:", f'{best_row["Precision"]:.4f}')
print("Recall:", f'{best_row["Recall"]:.4f}')
print("F1-score:", f'{best_row["F1-score"]:.4f}')
print("TP:", int(best_row["TP"]))
print("FP:", int(best_row["FP"]))
print("FN:", int(best_row["FN"]))

=== HASIL EVALUASI DATA VALIDASI ===


,Confidence,TP,FP,FN,Precision,Recall,F1-score
0,0.05,794,16052,417,0.047133,0.655656,0.087944
1,0.10,778,12364,433,0.059200,0.642444,0.108409
2,0.15,724,7862,487,0.084323,0.597853,0.147800
3,0.20,638,4573,573,0.122433,0.526837,0.198692
4,0.25,570,2681,641,0.175331,0.470685,0.255491
5,0.30,490,1523,721,0.243418,0.404624,0.303970
6,0.35,397,872,814,0.312845,0.327828,0.320161
7,0.40,334,521,877,0.390643,0.275805,0.323330
8,0.50,212,207,999,0.505967,0.175062,0.260123



=== AMBANG CONFIDENCE TERBAIK PADA VALIDASI ===
Confidence: 0.4
Precision: 0.3906
Recall: 0.2758
F1-score: 0.3233
TP: 334
FP: 521
FN: 877


In [2]:
# Muat ulang model SSD

import sys
import tensorflow as tf
from pathlib import Path
from google.protobuf import text_format

# 1. Tentukan lokasi proyek
PROJECT = Path("/home/rzangef/SSD-PPE")
RESEARCH_DIR = str(PROJECT / "object-detection-api/research")
CONFIG_PATH = PROJECT / "models/pipeline_ppe_yolo_split_val.config"
CHECKPOINT_PATH = PROJECT / "training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split/ckpt-7"

# 2. Periksa file yang diperlukan
assert CONFIG_PATH.exists(), f"Config tidak ditemukan: {CONFIG_PATH}"
assert Path(str(CHECKPOINT_PATH) + ".index").exists(), (
    f"Checkpoint tidak ditemukan: {CHECKPOINT_PATH}.index"
)

if RESEARCH_DIR not in sys.path:
    sys.path.insert(0, RESEARCH_DIR)

# 3. Impor modul Object Detection API
from object_detection.protos import pipeline_pb2
from object_detection.builders import model_builder

# 4. Baca konfigurasi model
pipeline_config = pipeline_pb2.TrainEvalPipelineConfig()

with open(CONFIG_PATH, "r") as f:
    text_format.Merge(f.read(), pipeline_config)

# 5. Bangun model untuk inferensi
detection_model = model_builder.build(
    model_config=pipeline_config.model,
    is_training=False
)

# 6. Pulihkan bobot dari checkpoint ckpt-7
checkpoint = tf.train.Checkpoint(model=detection_model)
restore_status = checkpoint.restore(str(CHECKPOINT_PATH))
restore_status.expect_partial()

print("=== MODEL BERHASIL DIMUAT KEMBALI ===")
print("Checkpoint:", CHECKPOINT_PATH)
print("TensorFlow:", tf.__version__)
print("Jumlah kelas:", pipeline_config.model.ssd.num_classes)

=== MODEL BERHASIL DIMUAT KEMBALI ===
Checkpoint: /home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split/ckpt-7
TensorFlow: 2.15.1
Jumlah kelas: 8


In [3]:
# Cek apakah model berhasil dimuat
from pathlib import Path

TEST_RECORD = Path(
    "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/test.record"
)

print("test.record tersedia:", TEST_RECORD.exists())
print("Ukuran file (MB):", round(TEST_RECORD.stat().st_size / (1024**2), 2)
      if TEST_RECORD.exists() else "File tidak ditemukan")

test.record tersedia: True
Ukuran file (MB): 19.99


In [4]:
# Evaluasi pada data test
import tensorflow as tf
import numpy as np
import pandas as pd
import time
from pathlib import Path

# Lokasi data test
TEST_RECORD = Path(
    "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/test.record"
)

# Pemeriksaan awal
assert "detection_model" in globals(), (
    "Model detection_model tidak tersedia. Muat checkpoint ckpt-7 terlebih dahulu."
)
assert TEST_RECORD.exists(), f"File tidak ditemukan: {TEST_RECORD}"

CONFIDENCE_THRESHOLD = 0.40
IOU_THRESHOLD = 0.50

FEATURES = {
    "image/encoded": tf.io.FixedLenFeature([], tf.string),
    "image/filename": tf.io.FixedLenFeature([], tf.string, default_value=b""),
    "image/object/bbox/xmin": tf.io.VarLenFeature(tf.float32),
    "image/object/bbox/xmax": tf.io.VarLenFeature(tf.float32),
    "image/object/bbox/ymin": tf.io.VarLenFeature(tf.float32),
    "image/object/bbox/ymax": tf.io.VarLenFeature(tf.float32),
    "image/object/class/label": tf.io.VarLenFeature(tf.int64),
}

def read_sparse(example, key, dtype):
    return tf.sparse.to_dense(example[key]).numpy().astype(dtype)

def compute_iou(box, boxes):
    # Format kotak: [ymin, xmin, ymax, xmax]
    if len(boxes) == 0:
        return np.array([], dtype=np.float32)

    y1 = np.maximum(box[0], boxes[:, 0])
    x1 = np.maximum(box[1], boxes[:, 1])
    y2 = np.minimum(box[2], boxes[:, 2])
    x2 = np.minimum(box[3], boxes[:, 3])

    intersection = np.maximum(0, y2-y1) * np.maximum(0, x2-x1)

    area1 = max(0, box[2]-box[0]) * max(0, box[3]-box[1])
    area2 = (
        np.maximum(0, boxes[:, 2]-boxes[:, 0]) *
        np.maximum(0, boxes[:, 3]-boxes[:, 1])
    )

    union = area1 + area2 - intersection
    return intersection / np.maximum(union, 1e-9)

# Tahap A: Inferensi seluruh gambar test
test_records = []
total_gt = 0
start_time = time.time()

dataset = tf.data.TFRecordDataset(str(TEST_RECORD))

for image_index, raw in enumerate(dataset):
    example = tf.io.parse_single_example(raw, FEATURES)

    image = tf.io.decode_image(
        example["image/encoded"],
        channels=3,
        expand_animations=False
    )
    image = tf.cast(image, tf.float32)[tf.newaxis, ...]

    preprocessed, shapes = detection_model.preprocess(image)
    prediction = detection_model.predict(preprocessed, shapes)
    detections = detection_model.postprocess(prediction, shapes)

    n = int(detections["num_detections"][0].numpy())

    pred_boxes = detections["detection_boxes"][0].numpy()[:n]
    pred_scores = detections["detection_scores"][0].numpy()[:n]

    # ID kelas foreground SSD diasumsikan 0-based dari model.
    # Tambahkan 1 agar sesuai dengan ID label_map 1–8.
    pred_classes = (
        detections["detection_classes"][0].numpy()[:n].astype(int) + 1
    )

    gt_classes = read_sparse(
        example, "image/object/class/label", np.int32
    )
    xmin = read_sparse(example, "image/object/bbox/xmin", np.float32)
    xmax = read_sparse(example, "image/object/bbox/xmax", np.float32)
    ymin = read_sparse(example, "image/object/bbox/ymin", np.float32)
    ymax = read_sparse(example, "image/object/bbox/ymax", np.float32)

    gt_boxes = np.column_stack([ymin, xmin, ymax, xmax]).astype(np.float32)

    if len(gt_classes) != len(gt_boxes):
        raise ValueError(
            f"Jumlah anotasi tidak cocok pada gambar ke-{image_index + 1}"
        )

    test_records.append({
        "pred_boxes": pred_boxes,
        "pred_scores": pred_scores,
        "pred_classes": pred_classes,
        "gt_boxes": gt_boxes,
        "gt_classes": gt_classes,
    })

    total_gt += len(gt_classes)

    if (image_index + 1) % 25 == 0:
        print(f"Sudah diproses: {image_index + 1} gambar")

print("\n=== INFERENSI TEST SELESAI ===")
print("Jumlah gambar:", len(test_records))
print("Jumlah ground truth:", total_gt)
print(f"Waktu inferensi: {(time.time()-start_time)/60:.1f} menit")

# Tahap B: Hitung TP, FP, FN pada threshold yang telah dipilih dari validasi
TP = FP = FN = 0

for rec in test_records:
    gt_boxes = rec["gt_boxes"]
    gt_classes = rec["gt_classes"]

    keep = rec["pred_scores"] >= CONFIDENCE_THRESHOLD
    boxes = rec["pred_boxes"][keep]
    scores = rec["pred_scores"][keep]
    classes = rec["pred_classes"][keep]

    order = np.argsort(-scores)
    boxes = boxes[order]
    classes = classes[order]

    matched_gt = set()
    image_tp = 0

    for box, pred_class in zip(boxes, classes):
        candidates = [
            j for j, gt_class in enumerate(gt_classes)
            if gt_class == pred_class and j not in matched_gt
        ]

        if not candidates:
            continue

        ious = compute_iou(box, gt_boxes[candidates])
        best_pos = int(np.argmax(ious))

        if ious[best_pos] >= IOU_THRESHOLD:
            matched_gt.add(candidates[best_pos])
            image_tp += 1

    TP += image_tp
    FP += len(boxes) - image_tp
    FN += len(gt_boxes) - image_tp

precision = TP / (TP + FP) if TP + FP else 0.0
recall = TP / (TP + FN) if TP + FN else 0.0
f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0.0
)

test_results = pd.DataFrame([{
    "Confidence": CONFIDENCE_THRESHOLD,
    "IoU": IOU_THRESHOLD,
    "Jumlah gambar": len(test_records),
    "Ground truth": total_gt,
    "TP": TP,
    "FP": FP,
    "FN": FN,
    "Precision": precision,
    "Recall": recall,
    "F1-score": f1,
}])

print("\n=== HASIL AKHIR EVALUASI TEST SSD MOBILENET ===")
display(test_results)
print(f"Precision : {precision:.4f} ({precision*100:.2f}%)")
print(f"Recall    : {recall:.4f} ({recall*100:.2f}%)")
print(f"F1-score  : {f1:.4f} ({f1*100:.2f}%)")

Sudah diproses: 25 gambar
Sudah diproses: 50 gambar
Sudah diproses: 75 gambar
Sudah diproses: 100 gambar
Sudah diproses: 125 gambar
Sudah diproses: 150 gambar
Sudah diproses: 175 gambar
Sudah diproses: 200 gambar
Sudah diproses: 225 gambar
Sudah diproses: 250 gambar
Sudah diproses: 275 gambar
Sudah diproses: 300 gambar
Sudah diproses: 325 gambar

=== INFERENSI TEST SELESAI ===
Jumlah gambar: 340
Jumlah ground truth: 2385
Waktu inferensi: 3.5 menit

=== HASIL AKHIR EVALUASI TEST SSD MOBILENET ===


,Confidence,IoU,Jumlah gambar,Ground truth,TP,FP,FN,Precision,Recall,F1-score
0,0.4,0.5,340,2385,579,992,1806,0.368555,0.242767,0.29272


Precision : 0.3686 (36.86%)
Recall    : 0.2428 (24.28%)
F1-score  : 0.2927 (29.27%)
